# Medical Guidelines RAG Chatbot & Evaluation Notebook

**Tech Stack:** Groq API (`openai` SDK + `openai/gpt-oss-120b`), Hugging Face `NeuML/pubmedbert-base-embeddings` (`sentence-transformers`), `ChromaDB`, `LangChain Text Splitters`, `PyPDF`

---
### Project Overview
In this notebook, we build a **Retrieval-Augmented Generation (RAG)** chatbot over three World Health Organization (WHO) clinical guideline PDFs:
1. **`9789241548397_eng.pdf`**: *Prevention and Control of Noncommunicable Diseases: Guidelines for primary health care in low-resource settings (2012)* — Type 2 Diabetes, Asthma, and COPD.
2. **`WHO-UCN-NCD-20.1-eng.pdf`**: *HEARTS-D: Diagnosis and Management of Type 2 Diabetes (2020)*.
3. **`ist pdf.pdf`**: *Guideline for the pharmacological treatment of hypertension in adults (2021)*.

### What We Will Do:
1. **Setup & Load PDFs**: Extract text page-by-page with metadata (filename and page number).
2. **Load Free Medical Embeddings from Hugging Face**: Use `NeuML/pubmedbert-base-embeddings` locally in Colab so we can embed hundreds of medical chunks in seconds with **zero API rate limits**.
3. **Build a Baseline RAG Chatbot (Groq)**: Split text into chunks, store PubMedBERT embeddings in **ChromaDB**, and generate answers with Groq (`openai/gpt-oss-120b` via the OpenAI-compatible SDK).
4. **Evaluate the Baseline Chatbot**: Use an **LLM-as-a-Judge** evaluation pipeline on a ground-truth test set measuring **Faithfulness (Groundedness)**, **Answer Relevance**, and **Completeness** (1–5 scale).
5. **Experiment & Improve RAG Performance**: Test three practical improvements:
   - **Experiment 1 (Chunking Strategy)**: Naive fixed-size chunks vs. Structure-aware recursive chunking with overlap.
   - **Experiment 2 (Query Expansion / Rewriting)**: Rewriting user questions into clinical search queries before retrieval.
   - **Experiment 3 (Context-Aware System Prompting & Top-K Tuning)**: Increasing `top_k` and instructing the model to reconcile updates across older (2012) and newer (2020/2021) WHO guidelines with source citations.
6. **Compare Results**: Visualize score improvements across all experiments in a summary table and bar chart.

## 🛠️ Step 1: Install Required Libraries & Load Models
Run the cell below to install `openai` (for Groq), `sentence-transformers` (for Hugging Face embeddings), `chromadb`, `pypdf`, and `langchain-text-splitters`.

In [ ]:
!pip install -q openai chromadb pypdf langchain-text-splitters pandas matplotlib sentence-transformers

 Configure Groq API Key & Load Hugging Face Medical Embedding Model
- **LLM Generation & Evaluation (Groq)**: Uses the standard `openai` library pointed at Groq (`https://api.groq.com/openai/v1`). Store your key in Colab Secrets as `GROQ_API_KEY` or paste it when prompted.
- **Active Groq Models**: Defaults to `openai/gpt-oss-120b` (or swap to `qwen/qwen3.6-27b` or `openai/gpt-oss-20b`).
- **Medical Embeddings (Local & Free)**: Uses **[NeuML/pubmedbert-base-embeddings](https://huggingface.co/NeuML/pubmedbert-base-embeddings)** from Hugging Face via `sentence-transformers`. Runs locally in Colab with zero API rate limits!

In [ ]:
import os
from getpass import getpass
from openai import OpenAI
from sentence_transformers import SentenceTransformer

# 1. Load Groq API Key (from Colab Secrets or input prompt)
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('Groq_API_Key')
except Exception:
    GROQ_API_KEY = None

if not GROQ_API_KEY:
    GROQ_API_KEY = os.environ.get('Groq_API_KEY') or getpass('Enter your Groq API Key: ')

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url='https://api.groq.com/openai/v1'
)

# Active Groq models: 'openai/gpt-oss-120b', 'qwen/qwen3.6-27b', or 'openai/gpt-oss-20b'
CHAT_MODEL = 'openai/gpt-oss-120b'

# Auto-verify that CHAT_MODEL is available on your Groq key
try:
    available_groq = [m.id for m in client.models.list().data]
    if CHAT_MODEL not in available_groq:
        for fallback in ['openai/gpt-oss-120b', 'qwen/qwen3.6-27b', 'openai/gpt-oss-20b']:
            if fallback in available_groq:
                CHAT_MODEL = fallback
                break
    print(f'✅ Groq Client initialized! Active LLM: {CHAT_MODEL}')
except Exception as e:
    print(f'⚠️ Could not list Groq models ({e}), proceeding with {CHAT_MODEL}')

# 2. Load Free Medical Embedding Model from Hugging Face
HF_EMBED_MODEL_NAME = 'NeuML/pubmedbert-base-embeddings'
print(f'⏳ Loading Hugging Face medical embedding model ({HF_EMBED_MODEL_NAME})...')
embed_model = SentenceTransformer(HF_EMBED_MODEL_NAME)
print(f'✅ Loaded {HF_EMBED_MODEL_NAME} successfully! (Embedding dimension: {embed_model.get_sentence_embedding_dimension()})')

##  Step 2: Load & Inspect the Three PDFs

- `9789241548397_eng.pdf` (WHO 2012 NCD Guidelines: Diabetes, Asthma, COPD)
- `WHO-UCN-NCD-20.1-eng.pdf` (WHO 2020 HEARTS-D Type 2 Diabetes)
- `ist pdf.pdf` (WHO 2021 Pharmacological Treatment of Hypertension)

In [ ]:
from pypdf import PdfReader
import pandas as pd

PDF_FILES = [
    '9789241548397_eng.pdf',
    'WHO-UCN-NCD-20.1-eng.pdf',
    'ist pdf.pdf'
]

def load_pdfs(pdf_paths):
    """Reads PDF files page by page and stores text + metadata."""
    documents = []
    for path in pdf_paths:
        if not os.path.exists(path):
            print(f'⚠️ Warning: {path} not found in current directory. Upload it to Colab files!')
            continue
        reader = PdfReader(path)
        for page_num, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ''
            text = ' '.join(text.split())
            if len(text) > 50:  # Skip blank/cover pages with almost no text
                documents.append({
                    'source': os.path.basename(path),
                    'page': page_num,
                    'text': text
                })
    return documents

raw_pages = load_pdfs(PDF_FILES)
print(f'✅ Loaded {len(raw_pages)} non-empty pages across {len(PDF_FILES)} PDFs.')

# Quick summary table of loaded pages per PDF
if raw_pages:
    df_pages = pd.DataFrame(raw_pages)
    display(df_pages.groupby('source').agg(total_pages=('page', 'count'), avg_chars=('text', lambda x: int(x.str.len().mean()))))

## 🤖 Step 3: Build the Baseline RAG Pipeline
For our **Baseline RAG**, we use a simple setup so we have a clear benchmark to improve upon:
- **Chunking**: Naive fixed character splitting (`chunk_size=500`, `chunk_overlap=0`).
- **Embeddings**: Hugging Face `NeuML/pubmedbert-base-embeddings` (local, fast, and medical-domain specific).
- **Vector Database**: `ChromaDB` in-memory collection.
- **Retrieval**: Top-2 chunks (`top_k=2`).
- **Prompt**: Basic instruction to answer from context using Groq.

In [ ]:
import chromadb
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

def get_embeddings(texts):
    """Generates medical domain embeddings locally using Hugging Face PubMedBERT."""
    embeddings = embed_model.encode(texts, batch_size=64, show_progress_bar=False)
    return embeddings.tolist()

def build_chroma_collection(pages, splitter, collection_name):
    """Splits pages into chunks, embeds them with PubMedBERT, and stores them in ChromaDB."""
    chroma_client = chromadb.Client()
    try:
        chroma_client.delete_collection(collection_name)
    except Exception:
        pass
    collection = chroma_client.create_collection(name=collection_name)

    chunks, metadatas, ids = [], [], []
    chunk_id = 0
    for doc in pages:
        split_texts = splitter.split_text(doc['text'])
        for chunk_text in split_texts:
            chunks.append(chunk_text)
            metadatas.append({'source': doc['source'], 'page': doc['page']})
            ids.append(f'chunk_{chunk_id}')
            chunk_id += 1

    print(f'⏳ Embedding {len(chunks)} chunks for [{collection_name}] using Hugging Face {HF_EMBED_MODEL_NAME}...')
    embeddings = get_embeddings(chunks)
    collection.add(documents=chunks, embeddings=embeddings, metadatas=metadatas, ids=ids)
    print(f'✅ Collection [{collection_name}] ready with {len(chunks)} chunks!')
    return collection

# 1. Build Baseline Collection (Fixed 500-char chunks, 0 overlap)
baseline_splitter = CharacterTextSplitter(separator=' ', chunk_size=500, chunk_overlap=0)
baseline_collection = build_chroma_collection(raw_pages, baseline_splitter, 'baseline_rag')

In [ ]:
import time

class SimpleResponse:
    def __init__(self, text):
        self.text = text

def generate_with_retry(prompt, system_instruction=None, temperature=0.2, response_mime_type=None):
    """Calls Groq API via OpenAI SDK with automatic retry on 429 rate limits."""
    messages = []
    if system_instruction:
        messages.append({'role': 'system', 'content': system_instruction})
    messages.append({'role': 'user', 'content': prompt})

    kwargs = {
        'model': CHAT_MODEL,
        'messages': messages,
        'temperature': temperature
    }
    if response_mime_type == 'application/json':
        kwargs['response_format'] = {'type': 'json_object'}

    for attempt in range(4):
        try:
            res = client.chat.completions.create(**kwargs)
            return SimpleResponse(res.choices[0].message.content)
        except Exception as e:
            if '429' in str(e) and attempt < 3:
                wait_s = 10 * (attempt + 1)
                print(f'⏳ Groq rate limit reached, waiting {wait_s}s...')
                time.sleep(wait_s)
            else:
                raise e

def retrieve_chunks(query, collection, top_k=2):
    """Embeds the user query with PubMedBERT and retrieves top_k matching chunks from ChromaDB."""
    q_emb = get_embeddings([query])[0]
    results = collection.query(query_embeddings=[q_emb], n_results=top_k)
    retrieved = []
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        retrieved.append({
            'text': doc,
            'source': meta['source'],
            'page': meta['page']
        })
    return retrieved

def baseline_rag_chat(question, collection=baseline_collection, top_k=2):
    """Simple Baseline RAG function."""
    retrieved = retrieve_chunks(question, collection, top_k=top_k)
    context_str = '\n\n'.join([f"Source ({c['source']}, p.{c['page']}): {c['text']}" for c in retrieved])

    prompt = f"""Answer the user's question using ONLY the context below.
If the answer is not in the context, say 'I do not have enough information in the provided documents.'

Context:
{context_str}

Question: {question}
Answer:"""

    response = generate_with_retry(prompt, temperature=0.2)
    return {
        'question': question,
        'answer': response.text,
        'retrieved_chunks': retrieved
    }

# Quick test of the baseline chatbot
sample_res = baseline_rag_chat('What is the first-line oral medication for type 2 diabetes?')
print('🤖 Baseline Answer:\n', sample_res['answer'])

## 📊 Step 4: Evaluate the Chatbot (LLM-as-a-Judge)
To know if our experiments actually improve the chatbot, we need:
1. **A Benchmark Test Set**: 6 realistic clinical questions spanning all three WHO PDFs, along with verified Ground Truth answers from the documents.
2. **An Automated Evaluator (LLM-as-a-Judge)**: Using Groq (`openai/gpt-oss-120b`) with JSON output mode to grade each response from **1 to 5** on three core RAG metrics:
   - **Faithfulness (1–5)**: Does the answer stick strictly to the retrieved context without hallucinating?
   - **Answer Relevance (1–5)**: Does it directly address the user's clinical question?
   - **Completeness / Accuracy vs. Ground Truth (1–5)**: Does it capture the key clinical facts, dosages, and thresholds in the reference answer?

In [ ]:
import json

EVAL_DATASET = [
    {
        'id': 'Q1',
        'question': 'What are the WHO diagnostic cut-off values for Diabetes using Fasting Plasma Glucose, 2-hour post-load venous plasma glucose, and HbA1c?',
        'ground_truth': 'Fasting plasma glucose >= 7.0 mmol/L (126 mg/dL), 2-hour post-load venous plasma glucose >= 11.1 mmol/L (200 mg/dL), and HbA1c >= 6.5% (48 mmol/mol). (2-hour post-load capillary plasma glucose is >= 12.2 mmol/L or 220 mg/dL).'
    },
    {
        'id': 'Q2',
        'question': 'How should severe hypoglycaemia in an unconscious diabetic patient (or plasma glucose <= 2.8 mmol/L) be treated in primary care?',
        'ground_truth': 'Administer 20 to 50 mL of 50% glucose (dextrose) intravenously over 1 to 3 minutes. If unavailable, substitute with any hypertonic glucose solution. Provide food as soon as the patient can ingest food safely.'
    },
    {
        'id': 'Q3',
        'question': 'What are the 5 steps in the stepwise treatment approach for managing stable asthma in primary health care?',
        'ground_truth': 'Step 1: Inhaled salbutamol as required (prn). Step 2: Continue salbutamol prn + add low-dose inhaled beclometasone (100-200 ug twice daily for adults, 100 ug once/twice daily for children). Step 3: Increase inhaled beclometasone to 200-400 ug twice daily. Step 4: Add low-dose oral theophylline (or increase inhaled beclometasone). Step 5: Add oral prednisolone in the lowest dose possible (usually <10 mg daily).'
    },
    {
        'id': 'Q4',
        'question': 'How are acute exacerbations of COPD managed regarding antibiotics, oral steroids, salbutamol, and oxygen concentration?',
        'ground_truth': 'Give antibiotics for COPD exacerbations with evidence of infection/purulent sputum; give a short course of oral prednisolone (30-40 mg daily for about 7 days) for acute severe exacerbations; give high doses of inhaled salbutamol via nebulizer or spacer; administer oxygen (if available) using a controlled device limiting concentration to 24%-28%.'
    },
    {
        'id': 'Q5',
        'question': 'According to the 2021 WHO Hypertension Guideline, at what blood pressure threshold should pharmacological treatment be initiated, and what are the three recommended first-line drug classes?',
        'ground_truth': 'Initiate treatment at confirmed SBP >= 140 mmHg or DBP >= 90 mmHg in all adults, and at SBP 130-139 mmHg for individuals with existing CVD or high CVD risk, diabetes, or CKD. The three first-line drug classes are: (1) thiazide and thiazide-like diuretics, (2) ACE inhibitors (ACEis) or angiotensin-receptor blockers (ARBs), and (3) long-acting dihydropyridine calcium channel blockers (CCBs).'
    },
    {
        'id': 'Q6',
        'question': 'Which antihypertensive medications are preferred during pregnancy, and which classes are strictly contraindicated?',
        'ground_truth': 'Preferred in pregnancy: methyldopa, beta-blockers (specifically labetalol), calcium channel blockers (specifically nifedipine, or verapamil), and direct-acting vasodilators (hydralazine). Contraindicated in pregnancy: Renin-angiotensin system inhibitors (ACE inhibitors, ARBs, direct renin inhibitors), spironolactone, and the beta-blocker atenolol.'
    }
]

def evaluate_rag_response(question, ground_truth, rag_output):
    """Uses Groq LLM as an impartial judge to score Faithfulness, Relevance, and Completeness (1-5)."""
    context_str = '\n'.join([f"- ({c['source']}, p.{c['page']}): {c['text']}" for c in rag_output['retrieved_chunks']])

    eval_prompt = f"""You are an impartial AI Evaluation Judge grading a medical RAG chatbot.
Evaluate the Chatbot Answer based on the Question, Ground Truth Reference, and Retrieved Context.

Question: {question}
Ground Truth Reference: {ground_truth}
Retrieved Context:
{context_str}

Chatbot Answer: {rag_output['answer']}

Score each metric as an integer from 1 (worst) to 5 (best):
1. faithfulness: 5 if every claim in the answer is supported by the Retrieved Context without hallucination; 1 if fabricated.
2. relevance: 5 if the answer directly and clearly addresses the question; 1 if off-topic or evasive.
3. completeness: 5 if the answer captures all key clinical details, dosages, and thresholds present in the Ground Truth Reference; 1 if missing most key facts.

Return ONLY valid JSON with keys: "faithfulness", "relevance", "completeness", "reasoning"."""

    response = generate_with_retry(
        eval_prompt,
        temperature=0.0,
        response_mime_type='application/json'
    )
    scores = json.loads(response.text)
    scores['overall'] = round((scores['faithfulness'] + scores['relevance'] + scores['completeness']) / 3.0, 2)
    return scores

def run_evaluation_suite(rag_fn, experiment_name):
    """Runs the RAG function across all test questions and compiles evaluation scores."""
    records = []
    print(f'🚀 Running evaluation for: {experiment_name}...')
    for item in EVAL_DATASET:
        rag_out = rag_fn(item['question'])
        eval_scores = evaluate_rag_response(item['question'], item['ground_truth'], rag_out)
        records.append({
            'experiment': experiment_name,
            'id': item['id'],
            'question': item['question'],
            'answer': rag_out['answer'],
            'faithfulness': eval_scores['faithfulness'],
            'relevance': eval_scores['relevance'],
            'completeness': eval_scores['completeness'],
            'overall': eval_scores['overall'],
            'reasoning': eval_scores['reasoning']
        })
    df_res = pd.DataFrame(records)
    print(f"✅ {experiment_name} Average Overall Score: {df_res['overall'].mean():.2f} / 5.00")
    return df_res

# Evaluate our Baseline RAG
df_baseline = run_evaluation_suite(baseline_rag_chat, '0_Baseline (Fixed 500, k=2)')
display(df_baseline[['id', 'faithfulness', 'relevance', 'completeness', 'overall', 'reasoning']])

## 🧪 Step 5: Experimenting to Improve Chatbot Responses
Why did the **Baseline RAG** miss details on multi-step protocols or lists (like the 5 asthma steps or full diagnostic tables)?
1. **Small chunks with zero overlap (`500` chars)** cut tables and numbered lists in half.
2. **Retrieving only `top_k=2` chunks** doesn't gather enough context when information spans multiple paragraphs.
3. **User phrasing** might not match exact clinical terminology in the WHO guidelines.

Let's test **three progressive experiments** to see how each change impacts our evaluation metrics!

### 🔬 Experiment 1: Structure-Aware Recursive Chunking + Overlap (`chunk_size=1000`, `overlap=200`, `top_k=4`)
Instead of cutting blindly every 500 characters, `RecursiveCharacterTextSplitter` splits on natural boundaries (paragraphs, sentences) and keeps a `200`-character overlap so tables and lists stay intact.

In [ ]:
# Build Improved Collection with RecursiveCharacterTextSplitter
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=['\n\n', '\n', '. ', ' ', '']
)
improved_collection = build_chroma_collection(raw_pages, recursive_splitter, 'recursive_rag')

def exp1_rag_chat(question):
    return baseline_rag_chat(question, collection=improved_collection, top_k=4)

df_exp1 = run_evaluation_suite(exp1_rag_chat, '1_Recursive_Chunking (1000/200, k=4)')
display(df_exp1[['id', 'faithfulness', 'relevance', 'completeness', 'overall', 'reasoning']])

### 🔬 Experiment 2: LLM Query Expansion (Clinical Search Rewriting)
Sometimes a user asks a short or informal question. In **Query Expansion**, we ask Groq to rewrite the user's question into a keyword-rich clinical search query (including synonyms like *FPG, HbA1c, salbutamol, beclometasone, ACEi, CCB*) before searching ChromaDB with PubMedBERT.

In [ ]:
def rewrite_clinical_query(user_question):
    """Expands a user question with clinical synonyms for better vector retrieval."""
    prompt = f"""Rewrite the following medical question into a concise, keyword-rich search query for retrieving WHO clinical guideline passages. Include relevant medical abbreviations (e.g., FPG, HbA1c, T2DM, COPD, ACEi, ARB, CCB, MDI). Return ONLY the rewritten query text.

User Question: {user_question}
Search Query:"""
    res = generate_with_retry(prompt, temperature=0.0)
    return res.text.strip()

def exp2_rag_chat(question):
    expanded_query = rewrite_clinical_query(question)
    retrieved = retrieve_chunks(expanded_query, improved_collection, top_k=4)
    context_str = '\n\n'.join([f"Source ({c['source']}, p.{c['page']}): {c['text']}" for c in retrieved])

    prompt = f"""Answer the user's question using ONLY the context below.
If the answer is not in the context, say 'I do not have enough information in the provided documents.'

Context:
{context_str}

Question: {question}
Answer:"""

    response = generate_with_retry(prompt, temperature=0.2)
    return {
        'question': question,
        'expanded_query': expanded_query,
        'answer': response.text,
        'retrieved_chunks': retrieved
    }

df_exp2 = run_evaluation_suite(exp2_rag_chat, '2_Query_Expansion + Recursive')
display(df_exp2[['id', 'faithfulness', 'relevance', 'completeness', 'overall', 'reasoning']])

### 🔬 Experiment 3: Clinical Prompt Engineering + Source Citation + `top_k=5`
Notice that our dataset has guidelines from **2012** (`9789241548397_eng.pdf`), **2020** (`WHO-UCN-NCD-20.1-eng.pdf`), and **2021** (`ist pdf.pdf`).
In **Experiment 3**, we:
1. Combine the **original question + expanded query** for retrieval (`top_k=5`) so we never lose exact phrase matches.
2. Provide a **Structured Clinical System Prompt** that instructs Groq to:
   - List exact dosages, thresholds, and all steps clearly using bullet points.
   - Prioritize newer guidelines (2020/2021) or note differences if both older and newer guidelines are retrieved.
   - Cite the exact PDF filename and page number `[Source, p. X]` for every clinical recommendation.

In [ ]:
CLINICAL_SYSTEM_PROMPT = """You are a WHO Clinical Guidelines Assistant.
Follow these rules strictly when answering:
1. Grounding: Use ONLY the provided Context passages. Never invent dosages or thresholds.
2. Completeness: Always include exact numerical cut-offs (e.g., mmol/L, mg/dL, mmHg, %), drug names, dosages, and all steps in a protocol if present in the context.
3. Guideline Updates: If both older (2012) and newer (2020 HEARTS-D or 2021 Hypertension) guidelines appear in context, prioritize the newer guideline while noting any relevant context.
4. Citations: Cite the source document and page number inline like [WHO-UCN-NCD-20.1-eng.pdf, p. 13] for key facts."""

def exp3_rag_chat(question):
    expanded_query = rewrite_clinical_query(question)
    combined_search = f"{question} {expanded_query}"
    retrieved = retrieve_chunks(combined_search, improved_collection, top_k=5)
    context_str = '\n\n'.join([f"[{c['source']}, p. {c['page']}]: {c['text']}" for c in retrieved])

    user_prompt = f"""Context Passages:
{context_str}

Clinical Question: {question}
Provide a clear, complete, well-structured answer with inline citations:"""

    response = generate_with_retry(
        user_prompt,
        system_instruction=CLINICAL_SYSTEM_PROMPT,
        temperature=0.1
    )
    return {
        'question': question,
        'answer': response.text,
        'retrieved_chunks': retrieved
    }

df_exp3 = run_evaluation_suite(exp3_rag_chat, '3_Clinical_Prompt + Hybrid_Query (k=5)')
display(df_exp3[['id', 'faithfulness', 'relevance', 'completeness', 'overall', 'reasoning']])

## 📈 Step 6: Compare All Experiments & Visualize Improvements
Let's aggregate the evaluation metrics across all four runs (**Baseline** vs. **Exp 1** vs. **Exp 2** vs. **Exp 3**) and plot the progression.

In [ ]:
import matplotlib.pyplot as plt

all_results = pd.concat([df_baseline, df_exp1, df_exp2, df_exp3], ignore_index=True)

summary_table = all_results.groupby('experiment')[['faithfulness', 'relevance', 'completeness', 'overall']].mean().round(2)
print('📊 Summary of Average Evaluation Scores (Out of 5.0):')
display(summary_table)

# Plot grouped bar chart of experiment scores
ax = summary_table.plot(kind='bar', figsize=(10, 5), rot=15, colormap='viridis', edgecolor='black')
plt.title('RAG Chatbot Evaluation: Baseline vs. Experiments (1-5 Scale)', fontsize=13, fontweight='bold')
plt.ylabel('Average Score (1 to 5)')
plt.ylim(0, 5.5)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.legend(loc='lower right')
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=8, padding=2)
plt.tight_layout()
plt.show()

## 💬 Step 7: Interactive Multi-Turn RAG Chatbot
Now that **Experiment 3** gives us our best, citation-backed responses, here is a clean interactive function with **conversation history** so you can ask follow-up questions about any of the three WHO PDFs.

In [ ]:
# chat_history = []

# def ask_medical_bot(user_question):
#     """Interactive RAG chat function with multi-turn memory and citations."""
#     history_str = ''
#     for turn in chat_history[-3:]:
#         history_str += f"User: {turn['user']}\nAssistant: {turn['bot']}\n\n"

#     expanded = rewrite_clinical_query(user_question)
#     retrieved = retrieve_chunks(f"{user_question} {expanded}", improved_collection, top_k=5)
#     context_str = '\n\n'.join([f"[{c['source']}, p. {c['page']}]: {c['text']}" for c in retrieved])

#     prompt = f"""Recent Conversation History:
# {history_str if history_str else 'None'}

# Retrieved WHO Guideline Context:
# {context_str}

# User Question: {user_question}
# Answer clearly based ONLY on the retrieved context and include [Filename, p. X] citations:"""

#     response = generate_with_retry(
#         prompt,
#         system_instruction=CLINICAL_SYSTEM_PROMPT,
#         temperature=0.1
#     )
#     answer = response.text
#     chat_history.append({'user': user_question, 'bot': answer})

#     print(f'🧑 User: {user_question}\n')
#     print(f'🤖 Chatbot:\n{answer}\n')
#     print('📚 Top Retrieved Sources:', [(c['source'], f"p.{c['page']}") for c in retrieved])
#     print('-' * 80)
#     return answer

# # Example multi-turn conversation:
# ask_medical_bot('When is metformin contraindicated in type 2 diabetes patients?')
# ask_medical_bot('What medication should be given instead if it is contraindicated?')